[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap06/cap06_aluno.ipynb)
[![](imagens/github-badge.png){width="20%"}](https://github.com/fzampirolli/pdi-vc)

# Inspeção Industrial e Análise de Documentos

🚧 **Capítulo em desenvolvimento**

O conteúdo deste capítulo encontra-se em fase de elaboração e será disponibilizado em versão futura, incluindo exercícios práticos, estudos de caso, desafios de programação e aplicações integradas.

---

Na **Parte I — Processamento Digital de Imagens (PDI)**, foram estudadas técnicas para transformação e aprimoramento de imagens, como operações morfológicas, filtragem espacial, convoluções, limiarização, segmentação e processamento no domínio da frequência.

A **Parte II — Visão Computacional (VC)** amplia esse escopo ao tratar da interpretação automática do conteúdo visual, envolvendo a extração de informações, o reconhecimento de padrões e a tomada de decisões a partir de imagens.

Este capítulo apresenta essa transição por meio de duas aplicações representativas:

1. **Inspeção Industrial Automatizada**, voltada ao controle de qualidade e à detecção de defeitos em linhas de produção;
2. **Análise Automatizada de Documentos**, aplicada ao processamento de formulários, avaliações e outros documentos estruturados por meio de sistemas de reconhecimento óptico de marcas (*Optical Mark Recognition* – OMR).

Essas aplicações integram técnicas de detecção de estruturas geométricas, extração de descritores invariantes, reconhecimento de padrões e classificação de objetos, constituindo a base de diversos sistemas modernos de inspeção visual e automação.

---

## Objetivos do Capítulo

Ao final deste capítulo, o estudante deverá ser capaz de:

* Aplicar técnicas de **alinhamento automático de documentos** utilizando a Transformada de Hough;
* **Detectar e segmentar elementos de interesse** com base em propriedades geométricas e descritores invariantes;
* **Extrair informações estruturadas** por meio da análise de padrões espaciais;
* **Implementar sistemas de reconhecimento óptico de marcas** (OMR) para correção automatizada de avaliações e formulários;
* **Avaliar a robustez de algoritmos** frente a ruídos, variações de iluminação e distorções geométricas;
* **Desenvolver soluções de Visão Computacional** aplicadas à inspeção industrial e à análise automatizada de documentos.

---

Este capítulo marca a transição do **Processamento Digital de Imagens**, voltado à transformação de imagens, para a **Visão Computacional**, cujo objetivo é interpretar o conteúdo visual e utilizá-lo na tomada de decisões automatizadas.


## Configuração do Ambiente

Os exemplos deste capítulo utilizam bibliotecas amplamente empregadas em
Processamento Digital de Imagens e Visão Computacional. O bloco abaixo
instala os pacotes necessários; em ambientes que já os possuam, a execução
pode ser ignorada.

In [ ]:
#| quarto-raw: true

# Instalar apenas as bibliotecas ausentes
import importlib
import subprocess
import sys

for mod, pkg in {
    "cv2": "opencv-python",
    "skimage": "scikit-image",
    "numpy": "numpy",
    "pdf2image": "pdf2image",
    "pandas": "pandas",
    "tabulate": "tabulate",
    "PyPDF2": "PyPDF2",
    "bcrypt": "bcrypt",
    "pyarrow": "pyarrow",
    "pyzbar": "pyzbar",
}.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

# Imports
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, data, color

Além dessas bibliotecas, será utilizado o módulo didático `morph.py`,
desenvolvido para simplificar operações de leitura, visualização e
processamento de imagens ao longo deste livro. O código a seguir verifica
sua disponibilidade, realiza o *download* quando necessário e confirma a
versão carregada.

In [ ]:
#| quarto-raw: true

import os
import urllib.request

if not os.path.exists("morph.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/morph.py",
        "morph.py",
    )

import morph
from morph import mm

print(f"✅ Ambiente pronto. morph {getattr(morph, '__version__', 'local_file')}")

## Bases de Imagens para Experimentação

Os exemplos desta parte do livro utilizam, sempre que possível, documentos digitalizados, folhas de respostas, códigos de barras, *QRCodes* e outras imagens provenientes de aplicações reais. Para facilitar a reprodução dos experimentos, também são empregadas imagens públicas amplamente utilizadas no ensino e na pesquisa em Visão Computacional.

### Imagens Públicas com `skimage.data`

A biblioteca `skimage.data` disponibiliza uma coleção de imagens de referência frequentemente utilizada em PDI-VC. O conjunto inclui fotografias, documentos digitalizados, padrões sintéticos e outros exemplos adequados para segmentação, extração de contornos, análise geométrica e reconhecimento de padrões.

A  @tbl-skimage-data apresenta algumas das principais imagens dessa coleção, enquanto a  @fig-06-skimage-data ilustra exemplos representativos. Neste capítulo, destacam-se as imagens `page()`, `text()` `coffee()` e `brick()`, particularmente adequadas para experimentos de OCR (*Optical Character Recognition*), OMR (*Optical Mark Recognition*) e análise documental.

### Imagens Reais com OpenCV

Embora as imagens do `skimage.data` sejam adequadas para demonstrações e validação de algoritmos, aplicações reais normalmente utilizam imagens obtidas por digitalização, câmeras ou outros dispositivos de aquisição.

Para esses casos, a biblioteca OpenCV (`cv2`) oferece funções para leitura de imagens, captura de vídeo e acesso a dispositivos de aquisição. Ao longo deste livro, essas funções são encapsuladas pelo módulo didático `morph.py`, por meio de rotinas como `mm.read`, proporcionando uma interface única para manipulação das imagens. Dessa forma, os algoritmos desenvolvidos podem ser aplicados tanto às imagens públicas quanto a documentos e cenas reais, preservando o mesmo fluxo de processamento.


In [ ]:
#| quarto-raw: true
#| label: tbl-skimage-data
#| tbl-cap: "Principais imagens públicas disponíveis no módulo *skimage.data*."
#| echo: false
#| output: asis

import pandas as pd
from IPython.display import Markdown

dados = {
    "Função": [
        "`data.astronaut()`",
        "`data.camera()`",
        "`data.cat()`",
        "`data.chelsea()`",
        "`data.clock()`",
        "`data.coins()`",
        "`data.coffee()`",
        "`data.horse()`",
        "`data.moon()`",
        "`data.page()`",
        "`data.text()`",
        "`data.checkerboard()`",
        "`data.binary_blobs()`"
    ],
    "Descrição": [
        "Fotografia colorida de um astronauta.",
        "Fotografia clássica em tons de cinza amplamente utilizada em PDI.",
        "Fotografia colorida de um gato.",
        "Retrato colorido da gata Chelsea.",
        "Relógio analógico para detecção de formas e contornos.",
        "Conjunto de moedas utilizado em segmentação e *watershed*.",
        "Fotografia colorida de uma xícara de café.",
        "Silhueta binária de um cavalo.",
        "Imagem da Lua em tons de cinza.",
        "Página digitalizada de documento.",
        "Imagem contendo texto impresso para experimentos de OCR.",
        "Padrão xadrez para calibração e transformações geométricas.",
        "Blobs binários sintéticos para estudos de conectividade e morfologia."
    ]
}

df = pd.DataFrame(dados)

Markdown(
    df.to_markdown(
        index=False,
        colalign=("left", "left")
    )
)

In [ ]:
#| label: fig-06-skimage-data
#| fig-cap: "Algumas imagens públicas disponíveis em *skimage.data*."
#| echo: true
#| output: true

import matplotlib.pyplot as plt
from skimage import data

imgs = {
    "camera": data.camera(),
    "coins": data.coins(),
    "text": data.text(),
    "page": data.page(),
    "moon": data.moon(),
    "cat": data.cat(),
    "horse": data.horse(),
    "binary_blobs": data.binary_blobs()
}

fig, ax = plt.subplots(2, 4, figsize=(10, 5))

for a, (nome, img) in zip(ax.ravel(), imgs.items()):
    a.imshow(img, cmap="gray")
    a.set_title(nome)
    a.axis("off")

plt.tight_layout()

## Fundamentos de OMR e Inspeção Industrial

O **Reconhecimento Óptico de Marcas** (*Optical Mark Recognition* — OMR) é uma técnica de Visão Computacional destinada à identificação automática de marcações em posições previamente definidas de um formulário. Suas aplicações incluem folhas de respostas, questionários, formulários administrativos e outros documentos estruturados.

Diferentemente do OCR (*Optical Character Recognition*), que reconhece caracteres e palavras, o OMR determina a presença, a ausência ou a intensidade de marcas em regiões previamente conhecidas. Em vez de interpretar texto, explora propriedades geométricas e estatísticas associadas ao preenchimento dessas regiões.

Os sistemas modernos de OMR processam imagens obtidas por *scanners*, câmeras ou dispositivos móveis, automatizando tarefas que anteriormente dependiam de equipamentos especializados.

De forma geral, um sistema de OMR compreende as seguintes etapas:

1. **Aquisição:** conversão do documento físico em formato digital;
2. **Pré-processamento:** correção geométrica, redução de ruídos e binarização;
3. **Localização das regiões de interesse:** identificação das áreas destinadas às marcações;
4. **Análise das marcações:** avaliação do preenchimento das regiões candidatas;
5. **Interpretação:** conversão das marcações em respostas ou dados estruturados.

Esses princípios se estendem naturalmente à **Inspeção Industrial Automatizada**.
Em linhas de produção, o mesmo encadeamento — aquisição, pré-processamento,
segmentação, extração de características e decisão — é empregado para
detectar defeitos superficiais, verificar a integridade de componentes e
medir dimensões com precisão subpixel. A diferença reside no domínio de
aplicação: enquanto o OMR opera sobre documentos com estrutura predefinida,
a inspeção industrial lida com objetos cujas variações geométricas e
radiométricas devem ser modeladas de forma mais flexível.

Nas seções seguintes, ambas as aplicações são desenvolvidas por meio de
projetos práticos que reproduzem etapas típicas de sistemas reais.

---

## Projetos Práticos: Construção de um *Pipeline* de Análise Documental

Os conceitos deste capítulo serão desenvolvidos por meio de projetos que reproduzem etapas típicas de sistemas reais de análise documental, introduzindo técnicas reutilizáveis em aplicações de OCR, OMR, inspeção visual e processamento de formulários.

### Alinhamento Automático de Documentos (*OCR/OMR Pre-processing*)

A correção de inclinação (*deskew*) é uma etapa fundamental no processamento de documentos. Rotações introduzidas durante a digitalização ou captura comprometem a localização de regiões de interesse e reduzem a precisão das etapas subsequentes.

Neste projeto será desenvolvido um sistema para estimar automaticamente a orientação predominante do documento e corrigir sua inclinação. Para isso, serão empregadas técnicas clássicas de detecção de bordas com o operador de Canny e detecção de retas pela Transformada de Hough. A partir das linhas identificadas, será estimado o ângulo de rotação e aplicada uma transformação afim para produzir uma versão alinhada do documento.

Como formulários e folhas de resposta são frequentemente distribuídos em formato PDF, o *pipeline* inicia-se com a rasterização de cada página, convertendo-a em uma imagem matricial. Neste capítulo, essa etapa será realizada com a biblioteca `pdf2image`, gerando imagens PNG com resolução de 300 DPI (*dots per inch*). A partir delas, poderão ser aplicadas as técnicas de detecção de bordas, Transformada de Hough, segmentação, extração de contornos e reconhecimento automático de padrões estudadas ao longo do capítulo.


In [ ]:
#| label: fig-06-pdf-para-imagem
#| fig-cap: "*Pipeline* de ingestão de documentos: rasterização adaptativa de páginas PDF para matrizes discretas em formato PNG, exibindo a página dois."
#| echo: true
#| output: true

from pdf2image import convert_from_path

# Diretório dos microdados e folhas de respostas do exame institucional
file_path = "dados/provas_qrcode_EP.pdf"
print(f"PDF de folhas de prova digitalizadas: {file_path}")

if os.path.exists(file_path):
    # Rasterização das páginas com resolução otimizada de 300 DPI
    pages = convert_from_path(file_path, dpi=300)
    for i, page in enumerate(pages):
        saida = f"test{i+1:02d}.png"
        page.save(saida)
        print(f"[INGESTÃO] Página PDF convertida com sucesso: {saida}")
else:
    print("[AVISO] Arquivo PDF não localizado no path. Ativando fallback via skimage.data.")
    # Injeta matriz de texto pública para garantir a execução contínua do pipeline
    img_fallback = data.text()
    cv2.imwrite("test02.png", img_fallback)
    print("[INGESTÃO] Imagem de fallback estruturada: test02.png")

# Carrega e exibe a imagem rasterizada inicial utilizando o ecossistema morph
img_original = mm.read('test02.png')
mm.show(img_original, figsize=(4, 3))

### Algoritmo de Retificação de Inclinação (*Deskew*)

A etapa de *deskew* tem como objetivo estimar e corrigir a inclinação global de um documento digitalizado. A  @fig-06-comparativo-pipeline apresenta o fluxo de processamento, desde a imagem original até o resultado após a correção geométrica.

O procedimento é composto por três etapas principais:

1. detecção de bordas com o operador de Canny;
2. estimação da orientação predominante pela Transformada de Hough;
3. correção da inclinação por meio de uma transformação afim de rotação.

Após a retificação, o documento fica alinhado aos eixos da imagem, favorecendo as etapas subsequentes de segmentação, extração de componentes conexos e reconhecimento de marcas.

### Modelagem Matemática

Esta seção apresenta os fundamentos matemáticos das três etapas que compõem o processo de retificação: detecção de bordas, estimação da orientação e correção geométrica.

#### Detecção de Bordas

Inicialmente, a imagem é suavizada por um filtro Gaussiano para reduzir ruídos. Os conceitos de filtragem espacial e convolução foram apresentados no **Capítulo 3**. Em seguida, o operador de Canny calcula a magnitude do gradiente,

$$
|\nabla f| =
\sqrt{
\left(\frac{\partial f}{\partial x}\right)^2 +
\left(\frac{\partial f}{\partial y}\right)^2
}.
$$

Após a supressão de não-máximos e a limiarização, obtém-se uma imagem binária contendo as principais bordas do documento.

#### Transformada de Hough

As bordas detectadas são analisadas pela Transformada de Hough Linear. Cada ponto $(x,y)$ contribui para o conjunto de retas descrito por

$$
\rho = x \cos\theta + y \sin\theta.
$$

Os máximos da matriz acumuladora correspondem às estruturas lineares predominantes, como bordas da página, linhas de formulários e linhas de texto.

Para estimar a orientação global, são considerados apenas os ângulos pertencentes ao intervalo $[-45^\circ,45^\circ]$. A mediana desses valores é utilizada como estimativa da inclinação, reduzindo a influência de detecções espúrias.

#### Rotação Afim

Conhecido o ângulo de inclinação, aplica-se uma transformação afim de rotação em torno do centro da imagem. As transformações afins foram estudadas no **Capítulo 2**, juntamente com as operações de translação, escala, cisalhamento e rotação. A função `mm.rotate` implementa essa transformação utilizando interpolação bicúbica para preservar a qualidade visual de contornos e caracteres. 



In [ ]:
#| label: fig-06-comparativo-pipeline
#| fig-cap: "*Pipeline* de retificação axial: exibição comparativa entre a entrada rotacionada original, o mapa de gradientes estruturais de Canny e o resultado final alinhado com fundo normalizado em branco."
#| echo: true
#| output: true

def retificar_inclinacao_documento(img):

    gray = mm.gray(img) if img.ndim == 3 else img
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5,5), 0), 50, 150)
    lines = cv2.HoughLines(edges, 1, np.pi/180, 200)

    if lines is None:
        return edges, img

    angulos = []
    for line in lines:
        angulo = np.rad2deg(line[0][1]) - 90
        if -45 < angulo < 45:
            angulos.append(angulo)

    if not angulos:
        return edges, img

    return edges, mm.rotate(img, np.median(angulos), interp="bicubic")

# Execução do pipeline de deskew
img_edges, img_final = retificar_inclinacao_documento(img_original)

# Exibição múltipla padronizada com o formato nativo do livro
mm.show(
    [img_original, img_edges, img_final],
    titles=["Imagem Original", "Bordas de Canny", "Documento Retificado"],
    cols=3,
    figsize=(12, 4)
)

::: {.callout-note}
## 🧠 Por que funciona? — Transformada de Hough

Na Transformada de Hough, cada pixel de borda contribui com votos para
todas as retas que podem passar por sua posição. Em vez de selecionar
apenas a reta com maior número de votos, o algoritmo considera todas as
retas cuja quantidade de votos excede um limiar mínimo e calcula seus
respectivos ângulos. A inclinação global do documento é então estimada
pela mediana desses ângulos, uma medida robusta a valores discrepantes.
Assim, retas espúrias produzidas por sombras, ruídos ou outros elementos
da imagem exercem pouca influência sobre a estimativa final, desde que a
maioria das retas detectadas corresponda às bordas do documento.
:::

### Limitações Práticas

Embora apresente bom desempenho em condições usuais de digitalização, o método depende da existência de estruturas lineares suficientemente definidas para serem detectadas pela Transformada de Hough, como bordas da página, linhas de formulários ou linhas de texto. Sua precisão pode ser reduzida em imagens com baixa resolução, ruído excessivo, sombras intensas ou grandes inclinações. Em geral, documentos digitalizados com resolução próxima de 300 DPI e iluminação homogênea fornecem resultados adequados para aplicações de OCR e OMR.

A implementação apresentada neste capítulo possui finalidade didática, ilustrando os princípios da correção automática de inclinação por meio da detecção de bordas, da Transformada de Hough e da rotação afim. Por utilizar apenas a orientação das estruturas lineares predominantes, o método pode ser aplicado a diferentes tipos de documentos, sem depender de marcadores específicos.

Em sistemas reais de análise documental, entretanto, o alinhamento normalmente utiliza marcadores geométricos previamente conhecidos. No modelo de folha de respostas empregado pelo ecossistema MCTest, por exemplo, são utilizados quatro discos pretos de referência, além das regiões correspondentes ao cabeçalho, ao *QRCode* e aos quadros de respostas. A localização desses elementos permite estimar simultaneamente a rotação, a escala e a translação da folha, tornando o registro menos sensível à quantidade de texto, à ausência de linhas estruturais e às variações de impressão ou digitalização.

Por esse motivo, a abordagem baseada na Transformada de Hough é utilizada neste capítulo para introduzir os fundamentos do problema, enquanto as etapas posteriores adotam o alinhamento por marcadores geométricos, estratégia predominante em sistemas de OMR e análise documental.


## Normalização de Fundo e Equalização Local de Contraste

A qualidade da segmentação depende diretamente do contraste da imagem de entrada. Em documentos digitalizados, variações de iluminação, sombras, regiões superexpostas e diferenças de tonalidade do papel dificultam a separação entre primeiro plano e fundo por meio de um limiar global.

Duas estratégias complementares são utilizadas para compensar essas variações:

- **Normalização de fundo:** divide a imagem por uma versão fortemente suavizada de si mesma (fundo estimado), cancelando gradientes de iluminação de baixa frequência — sombras, variação lateral de luz — antes da binarização.
- **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*), apresentada no **Capítulo 4:** divide a imagem em pequenas regiões (*tiles*) e equaliza o histograma de cada uma individualmente, limitando a amplificação do contraste para evitar o realce excessivo do ruído. É mais eficaz quando a iluminação é localmente heterogênea, mas o gradiente global já foi removido.

A @fig-06-clahe-page compara as duas abordagens sobre a imagem `page()` da biblioteca `skimage.data`. Cinco versões são exibidas: a imagem original, a imagem com fundo normalizado, a binarização direta por Otsu (sem pré-processamento, como referência), o resultado após normalização de fundo seguida de Otsu, e o resultado após CLAHE seguido de Otsu. Essa comparação permite visualizar tanto o efeito intermediário da normalização quanto a qualidade final da segmentação em cada estratégia, sendo a normalização de fundo seguida de Otsu a que produz a binarização mais limpa para este tipo de documento.

No projeto de retificação apresentado na seção anterior, a folha de respostas foi digitalizada em condições controladas, tornando suficiente a aplicação direta da limiarização. Em aplicações reais, entretanto, documentos são frequentemente capturados por *scanners* ou câmeras de dispositivos móveis, sob iluminação não uniforme. Nessas situações, a combinação de normalização do fundo, CLAHE e limiarização produz segmentações mais robustas, beneficiando as etapas subsequentes do *pipeline* de Visão Computacional.


In [ ]:
#| label: fig-06-clahe-page
#| fig-cap: "Efeito da normalização de fundo e do CLAHE na limiarização por Otsu: da esquerda para a direita, qualidade crescente de segmentação."
#| echo: true
#| output: true

import cv2
from skimage import data
from morph import mm

img = data.page()  # ou: img = mm.gray(img_final)  — com imagem da folha de prova

# ── Método 1: Normalização de fundo + Otsu ────────────────────────────────
# Estima o fundo com um filtro Gaussiano de sigma grande (variações lentas de luz)
# e divide pixel a pixel para cancelar o gradiente de iluminação
bg = cv2.GaussianBlur(img, (0, 0), sigmaX=25)
img_norm = cv2.divide(img, bg, scale=255)
img_norm_otsu = mm.threshold(img_norm)

# ── Método 2: CLAHE + Otsu ────────────────────────────────────────────────
# tileGridSize define o tamanho de cada região local (tile)
# clipLimit controla o teto de amplificação — valores altos aumentam contraste
# mas também amplificam ruído
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img)
img_clahe_otsu = mm.threshold(img_clahe)

# ── Método 0: Otsu direto (sem pré-processamento) — referência ───────────
img_otsu = mm.threshold(img)

mm.show(
    [img, img_otsu, img_norm, img_clahe_otsu, img_norm_otsu],
    titles=["Original", "Otsu direto", "Fundo normalizado", "CLAHE + Otsu", "Normaliz. fundo + Otsu"],
    cols=3,
    figsize=(14, 8)
)


::: {.callout-note}
## 🧠 Por que funciona? — Normalização de fundo vs. CLAHE

**Normalização de fundo:** ao dividir a imagem por uma versão fortemente suavizada
de si mesma, eliminam-se as variações lentas de luminosidade (gradiente de luz,
sombra de borda) sem afetar os detalhes finos — texto, linhas, bolhas.
O resultado é uma imagem com iluminação aproximadamente uniforme, onde o limiar
global de Otsu passa a funcionar bem em toda a página.

**CLAHE:** um histograma global equalizado "estica" os tons de toda a imagem de
uma vez — útil quando a iluminação é uniforme, mas problemático quando não é.
O CLAHE divide a imagem em pequenos blocos (*tiles*) e equaliza cada um
separadamente, com um limite máximo de amplificação (`clipLimit`) para não explodir
o ruído. É especialmente eficaz para realçar regiões subexpostas localmente,
mas não elimina gradientes globais — por isso, aplicá-lo após a normalização de
fundo tende a produzir resultados mais consistentes.
:::


### Detecção de Bordas e Contornos

A localização precisa das regiões de interesse é uma etapa essencial em
sistemas de OMR. No modelo de folha de respostas utilizado neste capítulo,
o cabeçalho e o quadro de respostas estão contidos em um retângulo virtual
delimitado por quatro discos pretos posicionados nos cantos. A identificação
desses marcadores permite localizar a região de interesse e corrigir
distorções geométricas introduzidas durante a aquisição da imagem.

O procedimento é composto por cinco etapas. Inicialmente, aplica-se um
**fechamento morfológico** (dilatação seguida de erosão), operação estudada
no **Capítulo 4**, utilizando um elemento estruturante em disco
(`mm.sedisk(33)`). Essa operação reduz pequenas descontinuidades e preserva
os discos de referência, tornando-os mais homogêneos. Em seguida, a imagem é
invertida (`mm.neg`), de modo que os discos passem a constituir componentes
claros sobre fundo escuro.

Na etapa seguinte, aplica-se a operação `mm.edgeoff` (**Capítulo 4**), que
remove componentes conectados às bordas da imagem, eliminando artefatos como
sombras de digitalização, marcas de corte e outros objetos espúrios nas
margens. Os componentes remanescentes são então analisados a partir de seus
contornos e filtrados por propriedades geométricas, como área e
circularidade, para identificar os discos candidatos. A implementação do
MCTest torna esse processo mais robusto ao selecionar, entre todos os
candidatos, os quatro cujos centros formam um retângulo com largura
compatível com a da imagem, reduzindo a ocorrência de falsos positivos.

Por fim, os centros dos quatro discos são ordenados espacialmente (superior
esquerdo, superior direito, inferior esquerdo e inferior direito) e
utilizados como pontos de controle em uma transformação de perspectiva
(*perspective warp*). Essa transformação retifica a imagem, produzindo uma
representação alinhada e com dimensões conhecidas, adequada às etapas
subsequentes de segmentação e reconhecimento.

As principais etapas desse *pipeline*, desde o processamento morfológico até
a imagem retificada, são ilustradas a seguir.

In [ ]:
#| label: fig-06-deteccao-marcadores
#| fig-cap: "Detecção dos discos marcadores, extração de contornos e retificação por transformação de perspectiva."
#| echo: true
#| output: true

import cv2
import numpy as np
from morph import mm

# img: imagem em escala de cinza da folha de prova
img = mm.gray(img_final)

# 1. Fechamento morfológico: preserva os discos escuros, removendo tudo menor que o disco
img_close = mm.close(img, mm.sedisk(41))

# 2. Inversão: discos escuros tornam-se componentes claros sobre fundo escuro
img_neg = mm.neg(img_close)

# 3. Remove componentes conectados que tocam a borda da imagem
img_edgeoff = mm.edgeoff(img_neg)

# 4. Extração dos contornos externos
contornos, _ = cv2.findContours(img_edgeoff, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 5. Filtragem por área e circularidade, mantendo apenas os 4 discos
centros = []
for c in contornos:
    area = cv2.contourArea(c)
    perimetro = cv2.arcLength(c, True)
    if area < 50 or perimetro == 0:
        continue
    circularidade = 4 * np.pi * area / (perimetro ** 2)
    if circularidade > 0.6:
        M = cv2.moments(c)
        cx, cy = M["m10"] / M["m00"], M["m01"] / M["m00"]
        centros.append((cx, cy))

# Verificação robusta: interrompe o pipeline com mensagem clara em vez de AssertionError
if len(centros) != 4:
    print(f"[AVISO] Esperado 4 discos marcadores, encontrado {len(centros)}.")
    print("  Verifique se a imagem é uma folha de respostas MCTest válida")
    print("  ou ajuste os parâmetros de circularidade e área mínima.")
    img_retificada = img  # fallback: preserva a imagem sem retificação
else:
    # 6. Ordenação dos centros: superior-esquerdo, superior-direito, inferior-esquerdo, inferior-direito
    pts = np.array(centros, dtype=np.float32)
    soma = pts.sum(axis=1)
    diff = pts[:, 0] - pts[:, 1]
    tl = pts[np.argmin(soma)]
    br = pts[np.argmax(soma)]
    tr = pts[np.argmax(diff)]
    bl = pts[np.argmin(diff)]
    pts_ordenados = np.array([tl, tr, bl, br], dtype=np.float32)

    # 7. Retificação por transformação de perspectiva (warp)
    largura, altura = 400, 400
    destino = np.array(
        [[0, 0], [largura, 0], [0, altura], [largura, altura]], dtype=np.float32
    )
    M_persp = cv2.getPerspectiveTransform(pts_ordenados, destino)
    img_retificada = cv2.warpPerspective(img, M_persp, (largura, altura))

    mm.show(
        [img_close, img_edgeoff, img_retificada],
        titles=["Fechamento (sedisk 41)", "edgeoff", "Retificada (warp)"],
        cols=3,
        figsize=(12, 4)
    )


::: {.callout-note}
## 🧠 Por que funciona? — Do fechamento morfológico à retificação

**Fechamento morfológico:** a dilatação seguida de erosão preenche pequenas
descontinuidades e suaviza os contornos dos objetos sem alterar
significativamente sua forma global. Quando aplicado com um elemento
estruturante grande (`sedisk(41)`), que elimina estruturas menores,
como textos, linhas do formulário e ruídos, preservando os objetos de maior
escala, entre eles os discos de referência da folha.

**Circularidade:** após o isolamento dos candidatos, a métrica
$C = \frac{4\pi A}{P^2}$ permite identificar quais componentes apresentam
forma aproximadamente circular. Seu valor é igual a 1 para um círculo
perfeito e diminui à medida que a forma se afasta da circularidade.
Assim, um limiar como $C > 0{,}6$ elimina a maior parte dos falsos
positivos sem a necessidade de um modelo de aprendizado.

**Transformação de perspectiva (*perspective warp*):** uma vez identificados
os quatro discos, seus centros são utilizados como pontos de controle para
estimar a transformação projetiva que relaciona a imagem capturada ao plano
ideal do documento. Essa transformação corrige distorções causadas pela
inclinação da câmera ou do scanner, preservando a colinearidade das retas e
produzindo uma imagem frontal com dimensões conhecidas, adequada às etapas
subsequentes de segmentação e reconhecimento.
:::

### Isolamento, Segmentação e Decodificação do *QRCode*

Após a retificação geométrica da folha de respostas, realiza-se a detecção e
a decodificação do *QRCode* presente no formulário. Esse marcador armazena
informações utilizadas pelo sistema de OMR (*Optical Mark Recognition*),
como a identificação do aluno, o código da prova e sua variação,
possibilitando a recuperação do gabarito correspondente no banco de dados.
Por segurança, essas informações são criptografadas antes da geração do
*QRCode*. Assim, a sequência decodificada corresponde a uma string
hexadecimal, cuja interpretação é realizada exclusivamente pelo sistema
MCTest. O procedimento é composto por três etapas: pré-processamento
morfológico, isolamento da região do *QRCode* e decodificação de seu
conteúdo.

Inicialmente, a imagem retificada em escala de cinza é binarizada por meio
da operação `mm.threshold`. Em seguida, aplica-se uma **abertura
morfológica** (erosão seguida de dilatação), estudada no **Capítulo 4**,
utilizando um elemento estruturante quadrado (`mm.sebox(2)`). Essa operação
remove pequenos ruídos e suaviza imperfeições sem comprometer a estrutura do
marcador. Por fim, a imagem é invertida (`mm.neg`), de modo que o
*QRCode* passe a constituir um componente claro sobre fundo escuro,
facilitando a extração de seus contornos.

A localização do *QRCode* é realizada pela análise dos contornos externos da
imagem binarizada. Entre os componentes detectados, seleciona-se aquele com
maior área e geometria aproximadamente quadrada, descartando os demais
elementos impressos da folha. Em seguida, a região correspondente é expandida
por uma pequena margem de segurança, garantindo a preservação integral do
marcador.

O *QRCode* é então extraído diretamente da imagem retificada em escala de
cinza, preservando sua qualidade radiométrica. Como essa região geralmente
apresenta dimensões reduzidas, aplica-se um redimensionamento com
interpolação cúbica, aumentando a resolução espacial e favorecendo a
identificação de seus módulos. A leitura é realizada pelo detector de
*QRCode* do OpenCV (`cv2.QRCodeDetector`), que recupera a sequência de
caracteres originalmente codificada.

O fluxo completo desse processamento, desde o pré-processamento morfológico
até a decodificação do *QRCode*, é ilustrado na
@fig-06-processamento-qrcode. O trecho exibido na saída corresponde apenas
ao início da string hexadecimal criptografada; sua interpretação completa é
realizada internamente pelo MCTest após a decodificação.

In [ ]:
#| label: fig-06-processamento-qrcode
#| fig-cap: "*Pipeline* de processamento do *QRCode*: limiarização, abertura morfológica, inversão e recorte final para decodificação."
#| echo: true
#| output: true

import cv2
import numpy as np
from morph import mm

# f: imagem retificada convertida para o tipo correto de 8 bits (0–255)
f = img_retificada.astype('uint8')

# 1. Limiarização: conversão da imagem em tons de cinza para binária
f_thresh = mm.threshold(f)

# 2. Abertura morfológica: elimina pequenos ruídos e suaviza o contorno dos blocos
f_open = mm.open(f_thresh, mm.sebox(2))

# 3. Inversão morfológica: módulos escuros tornam-se componentes claros sobre fundo escuro
f_inv = mm.neg(f_open)

# 4. Conversão segura para uint8 com escala 0–255
img_uint8 = (f_inv.astype(np.uint8) * 255) if f_inv.max() == 1 else f_inv.astype(np.uint8)

# 5. Detecção de contornos externos
contornos, _ = cv2.findContours(img_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if not contornos:
    raise ValueError("Nenhum contorno encontrado. Verifique o limiar ou a imagem de entrada.")

# 6. Filtragem pelo maior contorno com proporção aproximadamente quadrada
#    (aspect ratio entre 0.7 e 1.3 descarta retângulos alongados da folha)
def is_square_like(contorno, tol=0.3):
    x, y, w, h = cv2.boundingRect(contorno)
    ratio = w / h if h > 0 else 0
    return (1 - tol) <= ratio <= (1 + tol)

candidatos = [c for c in contornos if is_square_like(c)]

if not candidatos:
    raise ValueError(
        "Nenhum contorno quadrado encontrado. "
        "Verifique se o QRCode está presente na imagem ou ajuste a tolerância."
    )

# Seleciona o maior candidato quadrado por área de bounding box
maior_contorno = max(candidatos, key=lambda c: cv2.boundingRect(c)[2] * cv2.boundingRect(c)[3])
x, y, w, h = cv2.boundingRect(maior_contorno)

# 7. Expansão da bounding box com margem de segurança (evita truncamento do QR Code)
margem = 5
h_img, w_img = img_uint8.shape[:2]
x1 = max(x - margem, 0)
y1 = max(y - margem, 0)
x2 = min(x + w + margem, w_img)
y2 = min(y + h + margem, h_img)

# 8. Recorte da região de interesse a partir da imagem original (nítida, em cinza)
img_qrcode_final = img_retificada[y1:y2, x1:x2]

# 9. Ampliação para resolução mínima de decodificação (400 px no lado maior)
#    cv2.QRCodeDetector requer módulos com ao menos 3–4 px de largura para decodificar
#    com segurança; imagens menores que ~400 px tendem a falhar.
lado = max(img_qrcode_final.shape[:2])
escala = max(400 / lado, 1.0)
img_para_leitura = cv2.resize(
    img_qrcode_final, None,
    fx=escala, fy=escala,
    interpolation=cv2.INTER_CUBIC
)

# 10. Inicialização do detector nativo de QRCode do OpenCV
detector = cv2.QRCodeDetector()

# 11. Detecção geométrica e decodificação dos dados textuais
dados, pontos, qrcode_reto = detector.detectAndDecode(img_para_leitura)


# Visualização intermediária: progressão da binarização ao isolamento do QRCode
mm.show(
    [f_thresh, f_open, f_inv, img_qrcode_final],
    titles=["1. Limiarização", "2. Abertura morfológica", "3. Inversão", "Imagem final"],
    cols=3, figsize=(12, 4)
)

# Validação e saída dos metadados extraídos
if dados:
    print(f"QRCode decodificado com sucesso: \n{dados[:50]}...")
else:
    raise ValueError(
        "Falha na decodificação do QRCode. "
        "Verifique o limiar, as margens da região ou a qualidade da imagem."
    )

::: {.callout-note}
## 🧠 Por que funciona? — Isolamento e decodificação do *QRCode*

**Abertura morfológica:** diferentemente do fechamento, a abertura (erosão
seguida de dilatação) remove pequenos ruídos e protuberâncias sem alterar
significativamente a geometria dos objetos maiores. Assim, preserva a
estrutura do *QRCode* enquanto elimina componentes espúrios que poderiam
dificultar sua localização.

**Seleção por geometria:** o *QRCode* possui formato aproximadamente
quadrado ($w/h \approx 1$). A combinação desse critério com a seleção do
componente de maior área descarta linhas do formulário, textos e outros
elementos impressos, permitindo isolar o marcador sem o uso de modelos de
aprendizado.

**Redimensionamento antes da decodificação:** quando o *QRCode* ocupa poucos
pixels na imagem, seus módulos tornam-se difíceis de distinguir. O
redimensionamento com interpolação cúbica aumenta a resolução espacial da
região de interesse, facilitando a identificação dos padrões do código pelo
`cv2.QRCodeDetector` e tornando a decodificação mais robusta.
:::

### Exercício: Adaptação do *Pipeline* para Código de Barras

**Contexto:** O *pipeline* desenvolvido nas seções anteriores converte um PDF
em imagens, corrige a inclinação (*deskew*) e decodifica um *QRCode*.
Códigos de barras lineares (1D) também são amplamente empregados em
documentos institucionais, notas fiscais e etiquetas logísticas, seguindo um
fluxo de processamento bastante semelhante.

**Desafio:** Adapte esse *pipeline* para processar o arquivo
`dados/provas_barcode.pdf`, cujas páginas contêm códigos de barras lineares
no lugar de *QRCodes*. Para isso:

1. Rasterize o PDF com `pdf2image` utilizando resolução de 300 DPI;
2. Aplique a etapa de correção de inclinação (*deskew*) desenvolvida na seção anterior;
3. Utilize a biblioteca `pyzbar` (apresentada na próxima seção) para decodificar o código de barras;
4. Exiba a imagem retificada e a sequência de caracteres decodificada.

**Saída esperada:** A imagem retificada, seguida da string decodificada do
código de barras.

**Dica:** Ao contrário do *QRCode*, que é bidimensional, códigos de barras
lineares normalmente podem ser decodificados diretamente da imagem
rasterizada em 300 DPI, sem a necessidade de redimensionamento da região de
interesse.

### Decodificação de Código de Barras

Além de *QRCodes*, a biblioteca `pyzbar` permite decodificar diversas
simbologias de códigos de barras lineares (1D), como EAN-13 e Code 128.
O procedimento consiste em localizar o símbolo na imagem e interpretar a
sequência de barras e espaços, produzindo a cadeia de caracteres
correspondente.

A @fig-06-barcode-decode apresenta um exemplo de código de barras e o
resultado de sua decodificação.

In [ ]:
#| label: fig-06-barcode-decode
#| fig-cap: "Decodificação de código de barras linear com *pyzbar*: imagem de entrada e dados extraídos."
#| echo: true
#| output: true

import os
import numpy as np
from pyzbar.pyzbar import decode
from morph import mm

barcode_path = 'dados/barcode.png'

if os.path.exists(barcode_path):
    image = mm.read(barcode_path)
else:
    # Fallback sintético: gera um padrão de barras verticais que simula um Code-128
    print("[AVISO] Arquivo 'dados/barcode.png' não encontrado.")
    print("        Usando imagem sintética para demonstração do pipeline.")
    h, w = 100, 400
    img_synth = np.ones((h, w), dtype=np.uint8) * 255
    # Barras escuras em posições regulares (padrão simplificado)
    for x in range(20, w - 20, 8):
        if (x // 8) % 3 != 0:
            img_synth[:, x:x+4] = 0
    image = img_synth

mm.show(image)

barcodes = decode(image)

if barcodes:
    dados_bc = barcodes[0].data.decode("utf-8")
    tipo = barcodes[0].type
    print(f"Código de barras decodificado com sucesso [{tipo}]:\n{dados_bc}")
else:
    print("[INFO] Nenhum código de barras detectado na imagem.")
    print("       Em imagem sintética isso é esperado — substitua pelo arquivo real para decodificar.")


### O MCTest como estudo de caso: do protótipo ao sistema em produção

Até este ponto, cada etapa do *pipeline* foi implementada e analisada
isoladamente: correção da inclinação (*deskew*), detecção de marcadores,
retificação por perspectiva e leitura do *QRCode*. O **MCTest** integra
essas etapas em um sistema utilizado desde 2012 na UFABC para a correção
automatizada de avaliações de centenas de estudantes por semestre,
oferecendo suporte a diferentes modelos de folhas de resposta, gabaritos
individualizados e geração automática de relatórios de desempenho
[@zampirolli2023mctest].

A partir deste ponto, o objetivo deixa de ser implementar novos algoritmos e
passa a ser compreender como eles são integrados em uma aplicação real,
atendendo a requisitos de robustez, manutenção e escalabilidade.

Desenvolvido na UFABC e disponibilizado como software de código aberto, o
MCTest será utilizado como estudo de caso. Nas próximas seções, será
analisado o módulo de Visão Computacional, implementado no arquivo
`CVMCTest.py`, destacando como os conceitos apresentados neste capítulo são
organizados em um sistema completo.

#### Obtenção e Preparação do Módulo

O arquivo `CVMCTest.py` integra o sistema MCTest e depende de modelos e
configurações do *framework* Django, indisponíveis fora do ambiente Web.
Para utilizá-lo neste capítulo, o arquivo é obtido com `requests` e essas
dependências são removidas com `sed`, tornando o módulo autocontido.

In [ ]:
import requests
CVMCTest = requests.get("https://raw.githubusercontent.com/fzampirolli/mctest/master/exam/CVMCTest.py")
with open('CVMCTest.py', 'w') as writefile:
    writefile.write(CVMCTest.text)

Cada comando `sed` remove importações específicas do ambiente Django,
tornando o arquivo `CVMCTest.py` utilizável de forma independente neste capítulo.

In [ ]:
# remove linhas with "form django.", ...
!sed --in-place '/from django./d' CVMCTest.py
!sed --in-place '/from exam./d' CVMCTest.py
!sed --in-place '/from mctest./d' CVMCTest.py
!sed --in-place '/from student./d' CVMCTest.py
!sed --in-place '/from topic./d' CVMCTest.py
!sed --in-place '/from .models import VariationExam/d' CVMCTest.py

::: {.callout-note}
## 🧠 Por que remover as dependências do Django?

O arquivo `CVMCTest.py` foi desenvolvido para integrar uma aplicação Django
e, por isso, importa modelos de banco de dados e configurações do sistema
que não estão disponíveis neste capítulo. Essas dependências impedem a
importação do módulo, embora não sejam necessárias para executar as rotinas
de Visão Computacional.

Os comandos `sed` removem apenas essas importações, preservando a lógica de
processamento de imagens. Esse procedimento exemplifica uma prática comum de
engenharia de software: desacoplar um módulo de sua infraestrutura para
permitir sua reutilização em outros contextos.
:::

#### Extração da Área de Respostas

A função `getAnswerArea` encapsula as etapas de localização dos discos de
referência e retificação por perspectiva, retornando a região da folha que
contém os quadros de marcação, já alinhada e com dimensões fixas. A
@fig-06-mctest-img-original apresenta a imagem original em escala de cinza,
enquanto a @fig-06-mctest-answer-area mostra a área de respostas extraída.

In [ ]:
#| label: fig-06-mctest-img-original
#| fig-cap: "Imagem da folha de respostas em escala de cinza carregada a partir do PDF rasterizado."
#| echo: true
#| output: true

import os
from pdf2image import convert_from_path
from skimage import data as skdata
import cv2
from morph import mm

file = "dados/provas_qrcode_EP.pdf"
MYFILES = 'extra02.qrcode'

if os.path.exists(file):
    pages = convert_from_path(file, 200)  # dpi 100=min 500=max
    numPAGES = 0
    for page in pages:
        myfile0 = MYFILES + '_p' + str(numPAGES) + '.png'
        page.save(myfile0)
        numPAGES += 1
        print(f"[INGESTÃO] Página convertida: {myfile0}")
    pages.clear()
    img_color = mm.read(myfile0)
    img_inicial = mm.gray(img_color)
else:
    print("[AVISO] Arquivo 'dados/provas_qrcode.pdf' não encontrado.")
    print("        Usando imagem pública skimage.data.page() como substituto.")
    img_inicial = skdata.page()

mm.show(img_inicial)


In [ ]:
#| label: fig-06-mctest-answer-area
#| fig-cap: "Área de respostas extraída por *getAnswerArea*: região retificada contendo os quadros de marcação."
#| echo: true
#| output: true

import CVMCTest
countPage = 0
img_getAnswerArea = CVMCTest.cvMCTest.getAnswerArea(img_inicial, countPage)
mm.show(img_getAnswerArea)

**Nota de compatibilidade:** versões recentes do NumPy (≥ 2.0) removeram o alias `np.int0`. Caso `CVMCTest.py` utilize esse tipo, o comando abaixo aplica a correção diretamente no arquivo antes de recarregá-lo:

In [ ]:
!sed -i 's/box = np.int0(cv2.boxPoints(rect))/box = cv2.boxPoints(rect).astype(np.intp)/' ./CVMCTest.py

In [ ]:
import importlib
import CVMCTest

importlib.reload(CVMCTest)

#### Segmentação do *QRCode* e Localização dos Quadros

Após a extração da área de respostas, o MCTest executa duas etapas
preparatórias para a leitura das bolhas: `segmentQRcode` e `findSquares`.

**Segmentação do *QRCode*.** A função `segmentQRcode` isola o *QRCode* para
uma tentativa de decodificação. Em seguida, `getQRCode` retorna o
indicador `myFlagArea`, que informa se a área de respostas foi localizada
corretamente, e o dicionário `qr`, utilizado para armazenar os metadados da
prova. Caso a decodificação não seja bem-sucedida, o processamento das
respostas é mantido, mas a identificação do aluno, obtida a partir do
*QRCode*, permanece em branco no arquivo CSV de saída. Dessa forma, todas as
folhas são processadas e registradas, uma por linha, mesmo quando o
*QRCode* não pode ser interpretado. A @fig-06-mctest-qrcode-seg2 apresenta o
*QRCode* segmentado.

In [ ]:
#| label: fig-06-mctest-qrcode-seg2
#| fig-cap: "Região do *QRCode* isolada por *segmentQRcode* dentro da área de respostas retificada."
#| echo: true
#| output: true

import CVMCTest
imgQRcode = CVMCTest.cvMCTest.segmentQRcode(img_getAnswerArea, countPage)
mm.show(imgQRcode)

A função `CVMCTest.cvMCTest.decodeQRcode(imgQRcode)` descriptografa a *string*
hexadecimal e retorna o dicionário `qr` com os metadados da prova,
apresentado na próxima seção.


##### Localização dos Quadros de Respostas {.unnumbered}

A área de respostas retornada por `getAnswerArea` engloba o cabeçalho da
folha, incluindo o *QRCode*, e os quadros de respostas posicionados logo
abaixo. Como apenas essa segunda região é utilizada na leitura das bolhas,
o recorte `img_getAnswerArea[300:, :]` isola a região de interesse
processada pela função `findSquares` para localizar os quadros de respostas.


In [ ]:
#| label: fig-mctest-getAnswerArea-crop
#| fig-cap: "Recorte inferior da área de respostas, concentrando os quadros de bolhas a serem segmentados."
#| echo: true
#| output: true

img_getAnswerArea_aux = img_getAnswerArea[300:,:]
mm.show(img_getAnswerArea_aux)

Os principais campos do dicionário `qr`, retornado pelo código a seguir, são:

* **`date`:** identificador temporal da prova, composto pela data de geração e um *timestamp* interno do MCTest.
* **`idClassroom`, `idExam`, `idStudent`:** identificadores da turma, da prova e do aluno, utilizados para localizar o gabarito e registrar os resultados.
* **`term`:** período letivo da prova.
* **`stylesheet`:** folha de estilo utilizada na geração do formulário, definindo seu *layout*.
* **`var1` a `var5`:** número de questões dos níveis de dificuldade 1 a 5, conforme a configuração da prova.
* **`text`:** número de questões dissertativas.
* **`answer`:** número de alternativas por questão.
* **`numquest`:** número total de questões da prova.
* **`correct`, `dbtext`:** campos preenchidos após a correção automática, contendo o gabarito e informações adicionais das questões.
* **`variations`, `variant`:** indicam a existência de variações da prova e a versão atribuída ao estudante.

Esses metadados permitem ao MCTest identificar a prova e o estudante,
recuperar o gabarito correspondente e configurar as etapas subsequentes de
leitura e correção das respostas. Na prática, a função
`CVMCTest.cvMCTest.decodeQRcode(imgQRcode)` é chamada internamente por
`CVMCTest.cvMCTest.getQRCode(img, countPage)`, apresentada na próxima seção,
que integra a segmentação e a decodificação do *QRCode* em uma única etapa
do *pipeline*.


In [ ]:
myFlagArea, qr = CVMCTest.cvMCTest.getQRCode(img_inicial, countPage)
myFlagArea, qr

In [ ]:
rectSquares = CVMCTest.cvMCTest.findSquares(qr,img_getAnswerArea, countPage)
rectSquares

#### Leitura Automática das Respostas

As etapas desenvolvidas anteriormente são integradas no trecho de código a
seguir para processar cada quadro de respostas e compor as marcações do
estudante. Para cada quadro identificado em `rectSquares`, as funções
`setColumns` e `setLines` estimam, respectivamente, o número de alternativas
por questão e o número de questões, com base na distribuição espacial das
bolhas. Em seguida, `segmentAnswers` avalia o grau de preenchimento de cada
bolha e identifica a alternativa marcada. Por fim, `setAnswarsOneLine`
consolida as respostas de todos os quadros no campo `qr['answers']`,
produzindo uma única sequência de respostas. Quando o MCTest é utilizado de
forma independente, sem integração com seu banco de questões, essa sequência é
comparada ao gabarito presente na primeira página do PDF, que corresponde ao
modelo de prova sem enunciados adotado neste capítulo. A
@fig-mctest-answers apresenta o conteúdo final de `qr['answers']`, contendo
as respostas lidas automaticamente.


In [ ]:
#| label: fig-mctest-answers
#| fig-cap: "Respostas lidas automaticamente pelo MCTest após segmentação e classificação de todas as bolhas."
#| echo: true
#| output: true

testAnswers = []
if myFlagArea:
  
  imgQ_all = []

  for countSquare in range(len(rectSquares)):
      p1, p2 = rectSquares[countSquare]

      if True:
          imgQi = CVMCTest.cvMCTest.imgAnswers[p1[0]:p2[0], p1[1]:p2[1]]
          [NUM_COLUMNS, img] = CVMCTest.cvMCTest.setColumns(imgQi, countPage, countSquare)
          [NUM_LINES, img] = CVMCTest.cvMCTest.setLines(imgQi, countPage, countSquare)
          NUM_RESPOSTAS = NUM_COLUMNS
          NUM_QUESTOES = NUM_LINES

      imgQiNC = CVMCTest.cvMCTest.imgAnswers[p1[0]:p2[0], p1[1]:p2[1]]
      testAnswers.append(CVMCTest.cvMCTest.segmentAnswers(
          [imgQi, imgQiNC], countPage, countSquare, NUM_QUESTOES, qr

      ))

      imgQ_all.append(imgQiNC)

  qr = CVMCTest.cvMCTest.setAnswarsOneLine(testAnswers, qr)  # deixa as respostas de cada quadro em uma linha

mm.show(imgQ_all)
print(f"Respostas lidas das {len(qr['answers'].split(","))} questões: \n{qr['answers']}")

::: {.callout-tip}

## Conectando os pontos

O dicionário `qr['answers']` representa o resultado integrado das etapas
desenvolvidas neste capítulo, desde a rasterização e a retificação
geométrica até a decodificação do *QRCode* e a interpretação do
preenchimento das bolhas.

A principal diferença entre o protótipo implementado neste capítulo e o
MCTest está na engenharia de software: tratamento de erros, suporte a
múltiplos formatos, integração com banco de dados e interface Web. Os
algoritmos de Visão Computacional permanecem essencialmente os mesmos.

No modo de operação utilizado neste capítulo, a primeira página do PDF é
assumida como gabarito, enquanto as páginas subsequentes correspondem às
folhas de respostas dos estudantes. O MCTest compara automaticamente cada
folha com esse gabarito para calcular a pontuação.
:::

## Inspeção Industrial Automatizada

A inspeção visual automatizada constitui uma importante aplicação da Visão
Computacional. Em sistemas de inspeção, câmeras capturam imagens de peças ou
produtos, que são analisadas por algoritmos para verificar critérios de
qualidade previamente definidos. Dependendo da aplicação, essa abordagem
pode reduzir a variabilidade da inspeção humana e aumentar a produtividade
do processo.

Neste capítulo são ilustradas duas estratégias clássicas para detecção de
defeitos:

- **Subtração de imagens:** compara a imagem capturada com uma referência
  livre de defeitos. Regiões cuja diferença de intensidade excede um limiar
  são classificadas como anomalias. Essa abordagem requer iluminação e
  posicionamento consistentes entre as imagens.

- **Análise de textura:** avalia a uniformidade local da superfície sem
  depender de uma imagem de referência. Pode ser aplicada a materiais
  homogêneos, como tecidos, papéis e metais, nos quais irregularidades
  locais podem indicar defeitos.

As duas abordagens são ilustradas a seguir com imagens sintéticas baseadas
em `skimage.data`, permitindo reproduzir os experimentos sem dependência de
bases de dados externas.

### Referências e *Datasets* Públicos

Em aplicações reais, algoritmos de detecção de defeitos são frequentemente
avaliados em *datasets* públicos. Neste capítulo, utilizam-se imagens
sintéticas para garantir a reprodutibilidade dos exemplos, ver @fig-industrial-defeito, mas os seguintes
conjuntos de dados podem ser empregados para experimentação e comparação de
algoritmos:

- **MVTec *Anomaly Detection Dataset* (MVTec AD):** reúne imagens de objetos e
  texturas com defeitos anotados em nível de pixel
  [@bergmann2019mvtec]. Disponível em:
  <https://www.mvtec.com/company/research/datasets/mvtec-ad>

- ***Kolektor Surface-Defect Dataset *(KolektorSDD):** contém imagens de
  componentes industriais com defeitos superficiais anotados
  [@tabernik2020segmentation]. Disponível em:
  <https://www.vicos.si/resources/kolektorsdd/>

- **NEU *Surface Defect Database*:** disponibiliza imagens de superfícies de
  aço laminado contendo seis categorias de defeitos
  [@song2013noise]. Disponível em:
  <http://faculty.neu.edu.cn/songkechen/zh_CN/zdylm/263270/list/index.htm>

In [ ]:
#| label: fig-industrial-defeito
#| fig-cap: "Detecção de defeito por subtração de imagem: produto de referência, imagem com defeito simulado e máscara de anomalia detectada."
#| echo: true
#| output: true

import numpy as np
from skimage import data, color
from morph import mm

# Imagem de referência (produto sem defeito)
product_color = data.coffee()
product_gray = color.rgb2gray(product_color)

# Inserção de defeito simulado: arranhão escuro de 10×100 px
defect_image = np.copy(product_gray)
defect_image[100:110, 200:300] = 0.1

# Detecção por subtração e limiarização
difference = np.abs(product_gray - defect_image)
defect_threshold = 0.15          # ajustável conforme a aplicação
detected_defect = (difference > defect_threshold).astype(np.uint8) * 255

mm.show(
    [product_gray, defect_image, detected_defect],
    titles=["Referência", "Com defeito", "Defeito detectado"],
    cols=3,
    figsize=(12, 4)
)

status = "Defeito detectado." if detected_defect.any() else "Produto conforme."
print(status)

### Detecção de Defeitos por Análise de Textura

Na ausência de uma imagem de referência, a inspeção pode explorar a
**homogeneidade local da textura**. Superfícies uniformes, como metais
polidos, tecidos e papéis, apresentam baixa variância local. Defeitos como
riscos, bolhas ou manchas aumentam essa variância em sua vizinhança,
permitindo sua detecção.

A @fig-industrial-textura ilustra essa abordagem com a imagem
`skimage.data.brick()`. Inicialmente, calcula-se a variância local em uma
janela deslizante; em seguida, o mapa resultante é limiarizado para destacar
as regiões heterogêneas.

In [ ]:
#| label: fig-industrial-textura
#| fig-cap: "Detecção de heterogeneidade de textura: mapa de variância local e máscara de anomalia."
#| echo: true
#| output: true

import numpy as np
import cv2
from skimage import data as skdata
from morph import mm

# Imagem de textura uniforme (tijolo)
texture = skdata.brick().astype(np.float32) / 255.0

# Inserção de defeito sintético: mancha clara 20×80 px
texture_defect = np.copy(texture)
texture_defect[60:80, 80:160] = 0.95

# Mapa de variância local (janela 15×15)
def variancia_local(img, ksize=15):
    img_f = img.astype(np.float32)
    mean  = cv2.blur(img_f, (ksize, ksize))
    mean2 = cv2.blur(img_f ** 2, (ksize, ksize))
    return np.clip(mean2 - mean ** 2, 0, None)

var_ref    = variancia_local(texture)
var_defect = variancia_local(texture_defect)
diff_var   = np.abs(var_defect - var_ref)

# Normaliza e limiariza
diff_norm = (diff_var / diff_var.max() * 255).astype(np.uint8)
_, mask   = cv2.threshold(diff_norm, 30, 255, cv2.THRESH_BINARY)

mm.show(
    [texture, texture_defect, diff_norm, mask],
    titles=["Textura original", "Com defeito", "Δ variância local", "Anomalia detectada"],
    cols=4,
    figsize=(16, 4)
)

status = "Defeito de textura detectado." if mask.any() else "Superfície conforme."
print(status)


::: {.callout-note}
## 🧠 Por que funciona? — Detecção de defeitos por subtração

A subtração pixel a pixel entre a imagem de referência e a imagem
capturada produz valores próximos de zero em regiões idênticas e valores
elevados onde há diferenças. A operação `np.abs` torna o método
insensível ao sinal da diferença, permitindo detectar tanto defeitos mais
claros quanto mais escuros que a referência. Em seguida, a limiarização
converte o mapa de diferenças em uma decisão binária: região conforme ou
defeituosa.

O principal parâmetro do método é o limiar (`defect_threshold`). Valores
muito baixos tornam o algoritmo sensível ao ruído e aumentam a ocorrência
de falsos positivos, enquanto valores muito altos podem impedir a detecção
de defeitos sutis.
:::

## Exercícios Práticos

Os exercícios a seguir consolidam os conceitos apresentados neste capítulo,
propondo extensões do *pipeline* de processamento documental desenvolvido ao
longo do texto.

### Exercício 1: Validação Robusta da Leitura

**Contexto:** A detecção das bolhas utiliza critérios geométricos e de
preenchimento. Em imagens de baixa qualidade, ruídos e manchas podem gerar
falsos positivos.

**Desafio:**

1. Investigue critérios adicionais para validar as bolhas detectadas;
2. Avalie diferentes limiares de preenchimento;
3. Compare os resultados em imagens com diferentes níveis de ruído;
4. Discuta o impacto das escolhas na taxa de acertos e de falsos positivos.

**Saída esperada:** Uma tabela ou gráfico comparando o desempenho para
diferentes limiares e exemplos de detecções corretas e incorretas.

---

### Exercício 2: Detecção de Marcação Inválida

**Contexto:** Em avaliações reais, uma questão pode apresentar dupla
marcação, rasuras ou ausência de resposta.

**Desafio:**

1. Detecte questões com duas ou mais alternativas preenchidas;
2. Identifique respostas em branco;
3. Defina um critério para sinalizar preenchimentos ambíguos;
4. Gere um relatório indicando o estado de cada questão.

**Saída esperada:** Um relatório em JSON ou `pandas.DataFrame` contendo,
para cada questão, a alternativa lida e seu estado (`OK`, `BRANCO`,
`DUPLA MARCAÇÃO` ou `SUSPEITA`).

---

### Exercício 3: *Pipeline* Completo

**Contexto:** As etapas desenvolvidas neste capítulo podem ser integradas em
uma única aplicação.

**Desafio:**

1. Implemente uma função `processar_prova(caminho_pdf)` que:
   - converta o PDF em imagem;
   - retifique a folha;
   - decodifique o *QRCode*;
   - leia as respostas;
   - retorne os metadados e as respostas do estudante.

2. Avalie o *pipeline* em diferentes condições de aquisição (rotação,
resolução e qualidade de digitalização).

3. Documente as principais limitações observadas e proponha possíveis
melhorias.

**Saída esperada:** Um programa reutilizável acompanhado de exemplos de
execução em diferentes folhas de resposta.

---

## Próximos Passos

Neste capítulo foi desenvolvido um *pipeline* completo para análise de
documentos, desde a conversão de PDFs em imagens até a leitura automática
de folhas de resposta por OMR. As técnicas estudadas — operações
morfológicas, transformações geométricas, análise de contornos e
decodificação de marcadores — constituem a base de diversos sistemas de
Visão Computacional.
Os próximos capítulos ampliam esse escopo para problemas mais gerais de
reconhecimento visual.